# Chapter 7 Lab — Self-Attention From Scratch

Implements scaled dot-product attention and a minimal multi-head block in plain NumPy, then
compares against `torch.nn.MultiheadAttention`.

## 1. Scaled dot-product attention in NumPy

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores)
    return weights @ V, weights

## 2. Visualize attention weights for a toy sentence

In [ ]:
import matplotlib.pyplot as plt

tokens = ["the", "animal", "didn't", "cross", "the", "street", "because", "it", "was", "tired"]
np.random.seed(0)
d = 8
X = np.random.randn(len(tokens), d)
Wq, Wk, Wv = (np.random.randn(d, d) * 0.3 for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
out, weights = attention(Q, K, V)

plt.figure(figsize=(6, 5))
plt.imshow(weights, cmap="viridis")
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.yticks(range(len(tokens)), tokens)
plt.title("Attention weights (untrained, random projection — illustrative)")
plt.colorbar()
plt.tight_layout()
plt.show()

## 3. Multi-head attention block (NumPy)

In [ ]:
def multi_head_attention(X, n_heads, d_model):
    d_k = d_model // n_heads
    heads_out = []
    for h in range(n_heads):
        Wq, Wk, Wv = (np.random.randn(d_model, d_k) * 0.3 for _ in range(3))
        Qh, Kh, Vh = X @ Wq, X @ Wk, X @ Wv
        out_h, _ = attention(Qh, Kh, Vh)
        heads_out.append(out_h)
    concat = np.concatenate(heads_out, axis=-1)
    Wo = np.random.randn(d_model, d_model) * 0.3
    return concat @ Wo

mha_out = multi_head_attention(X, n_heads=2, d_model=d)
print(mha_out.shape)

## 4. Compare against PyTorch's production implementation

In [ ]:
import torch, torch.nn as nn

torch_mha = nn.MultiheadAttention(embed_dim=d, num_heads=2, batch_first=True)
x_t = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
out_t, attn_weights_t = torch_mha(x_t, x_t, x_t)
print("torch output shape:", out_t.shape, "attn weights shape:", attn_weights_t.shape)

## Exercise

Remove the `/ np.sqrt(d_k)` scaling from `attention()` and re-run the visualization with a much
larger `d` (e.g. 512). What happens to the softmax distribution, and why does that hurt
gradient flow during training?